<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_LightGBM_without_lag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

Cloning into 'ML_fx'...
remote: Enumerating objects: 3252, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 3252 (delta 1), reused 5 (delta 1), pack-reused 3243 (from 2)
Receiving objects: 100% (3252/3252), 48.03 MiB | 8.71 MiB/s, done.
Resolving deltas: 100% (3166/3166), done.
Updating files: 100% (3120/3120), done.


In [2]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())


   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
1    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
2    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
3    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
4    5347.45      192.0       24.6  ...    20  151315           0.0  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

In [3]:
split_date = '2011-12-01'
train.drop(columns=['sales_lag_52'], inplace=True)


# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")

Final Training Set Shape: (279085, 23)
Validation Set Shape: (142485, 23)
Validation Period: 2011-12-02 to 2012-10-26


In [4]:
!pip install wandb -q
!pip install lightgbm

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [8]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import wandb
import os
import joblib
from sklearn.metrics import mean_absolute_error

# Initialize W&B run
run = wandb.init(
    project="ML_fx_LightGBM_Walmart",
    name="LightGBM_run3",
    config={
        'learning_rate': 0.1,
        'num_leaves': 31,
        'max_depth': -1,
        'n_estimators': 500,
        'early_stopping_rounds': 20,
        'seed': 42,
        'verbosity': -1
    }
)

params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'max_depth': -1,
    'n_estimators': 500,
    'seed': 42,
    'verbosity': -1
}

model = lgb.LGBMRegressor(**params)

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.log_evaluation(period=0)]
    )

    print(f"Best Iteration: {model.best_iteration_}")

    # Debug: Print available keys in evals_result_
    print("Evals Result Keys:", model.evals_result_.keys())

    val_maes = None

    # Try standard keys
    if 'valid_0' in model.evals_result_:
        valid_0_data = model.evals_result_['valid_0']
        print("Valid_0 Metrics:", valid_0_data.keys())

        # Check for 'mae', 'l1', or 'MAE'
        if 'mae' in valid_0_data:
            val_maes = valid_0_data['mae']
        elif 'l1' in valid_0_data:
            val_maes = valid_0_data['l1']
        elif 'MAE' in valid_0_data:
            val_maes = valid_0_data['MAE']

    if val_maes is not None:
        run.summary["best_val_mae"] = min(val_maes)
    else:
        print("Warning: Could not find MAE metrics in evals_result_")
        run.summary["best_val_mae"] = None

except Exception as e:
    print(f"Error during fitting: {e}")
    raise e

# ---------------------------------------------------------
# EVALUATION (Same as before)
# ---------------------------------------------------------
y_pred = model.predict(X_val)
y_train_pred = model.predict(X_train)

is_holiday = X_val['IsHoliday'].values
is_holiday_train = X_train['IsHoliday'].values
weights = np.where(is_holiday, 5, 1)
weights_train = np.where(is_holiday_train, 5, 1)

final_wmae = np.average(np.abs(y_val - y_pred), weights=weights)
final_mae = mean_absolute_error(y_val, y_pred)

train_wmae = np.average(np.abs(y_train - y_train_pred), weights=weights_train)
train_mae = mean_absolute_error(y_train, y_train_pred)

wandb.log({"final_validation_wmae": final_wmae, "final_validation_mae": final_mae})
wandb.log({"train_wmae": train_wmae, "train_mae": train_mae})

# Feature Importance
importance = model.feature_importances_
feature_names = X_train.columns.tolist()
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
importance_df = importance_df.sort_values(by='importance', ascending=False)

table = wandb.Table(dataframe=importance_df.head(20))
wandb.log({"feature_importance_table": table})

joblib.dump(model, "model_lgb.joblib")

# 3. Create Artifact
artifact = wandb.Artifact("lightgbm-model-3", type="model")
artifact.add_file("model_lgb.joblib")
run.log_artifact(artifact)

# 4. Register in Model Registry
# Format: "entity/project/artifact_name" -> "entity/project/registry_collection_name"
# Replace 'your_entity' with your W&B username/team name
run.link_artifact(
    artifact=artifact,
    target_path="slomi23-free-university-of-tbilisi-/ML_fx_LightGBM_Walmart/lightgbm-models"
)
print(f"Final Train WMAE: {train_wmae:.2f}")
print(f"Final Validation WMAE: {final_wmae:.2f}")

wandb.finish()


final_validation_mae,▁
final_validation_wmae,▁
train_mae,▁
train_wmae,▁
best_val_mae,3300.03949
final_validation_mae,3300.03949
final_validation_wmae,3325.9814
train_mae,2506.30201
train_wmae,2630.72841


Best Iteration: 0
Evals Result Keys: dict_keys(['valid_0'])
Valid_0 Metrics: odict_keys(['l1'])


wandb: WARNING Artifact "lightgbm-model-3" already exists with the same content. No new version will be created.


Final Train WMAE: 2630.73
Final Validation WMAE: 3325.98


final_validation_mae,▁
final_validation_wmae,▁
train_mae,▁
train_wmae,▁
best_val_mae,3300.03949
final_validation_mae,3300.03949
final_validation_wmae,3325.9814
train_mae,2506.30201
train_wmae,2630.72841
